# 🔗 Análise de Redes Complexas — Malha Logística
## Caracterização Estrutural do Grafo de Ocorrências por Rota

Este notebook implementa a análise de redes complexas descrita no Capítulo 3 da dissertação.  
A abordagem modela as rotas logísticas como um **grafo ponderado direcionado**, onde:

- **Nós** representam cidades (origens e destinos)
- **Arestas** conectam pares (origem → destino) com peso proporcional à frequência de ocorrências
- **Atributos de aresta** incluem o código de ocorrência histórica (`historico_cod_ocor`) predominante

### Pipeline de Análise
1. Pré-processamento e construção do grafo bipartido e do grafo de rotas
2. Estatísticas descritivas da rede
3. Detecção de comunidades (Algoritmo de Louvain)
4. Métricas de centralidade (grau, intermediação, proximidade, autovetor)
5. Análise de distribuição de grau (escala log-log / comportamento scale-free)
6. Visualizações interativas e estáticas
7. Exportação das features de rede para enriquecimento do dataset


## 1. Dependências

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
from collections import Counter

import community.community_louvain as community_louvain

# Configurações visuais
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
PALETTE = sns.color_palette("tab10")

print("✅ Dependências carregadas")


## 2. Carregamento e Pré-processamento dos Dados

In [ ]:
# ── Carregamento ──────────────────────────────────────────────────────────────
df_raw = pd.read_csv('banco_ia.csv', low_memory=False)

# ── Filtro: apenas registros com ocorrência histórica relevante ───────────────
df = df_raw[df_raw['historico_cod_ocor'] != 0].copy()
df['historico_cod_ocor'] = df['historico_cod_ocor'].astype(int)

print(f"Total de registros no banco    : {len(df_raw):>10,}")
print(f"Registros com cod_ocor != 0    : {len(df):>10,}")
print(f"Proporção filtrada             : {len(df)/len(df_raw)*100:.2f}%")
print()
print("Distribuição de historico_cod_ocor:")
print(df['historico_cod_ocor'].value_counts().rename_axis('cod').reset_index(name='freq').to_string(index=False))


In [ ]:
# ── Parse de origem/destino a partir de ref_rota ──────────────────────────────
# ref_rota tem formato "ORIGEM-DESTINO"; usa cidade_origem_prestacao como âncora
# para lidar corretamente com nomes de cidade que contêm hífen (ex: GUARDA-MOR)

def extrair_destino(row):
    rota  = str(row['ref_rota'])
    origem = str(row['cidade_origem_prestacao'])
    prefixo = origem + '-'
    if rota.startswith(prefixo):
        return rota[len(prefixo):]
    # fallback: split no primeiro hífen
    partes = rota.split('-', 1)
    return partes[1] if len(partes) == 2 else rota

df['origem']  = df['cidade_origem_prestacao'].str.strip().str.upper()
df['destino'] = df.apply(extrair_destino, axis=1).str.strip().str.upper()

print(f"Cidades de origem  únicas : {df['origem'].nunique()}")
print(f"Cidades de destino únicas : {df['destino'].nunique()}")
print(f"Rotas únicas              : {df['ref_rota'].nunique()}")
print()
print(df[['ref_rota','origem','destino','historico_cod_ocor']].drop_duplicates().head(10).to_string(index=False))


## 3. Construção dos Grafos

São construídos dois grafos complementares:

| Grafo | Nós | Arestas | Uso |
|---|---|---|---|
| **Bipartido** (`G_bip`) | Rotas (`ref_rota`) ↔ Ocorrências (`historico_cod_ocor`) | Co-ocorrência | Análise estrutural bipartida |
| **Logístico** (`G_log`) | Cidades | Origem → Destino ponderada | Centralidades, comunidades, scale-free |


In [ ]:
# ═══════════════════════════════════════════════════════
# GRAFO 1 — BIPARTIDO: ref_rota × historico_cod_ocor
# ═══════════════════════════════════════════════════════
G_bip = nx.Graph()

# Nós tipo A: rotas (bipartite=0)
rotas_unicas = df['ref_rota'].unique()
G_bip.add_nodes_from(rotas_unicas, bipartite=0, tipo='rota')

# Nós tipo B: códigos de ocorrência (bipartite=1)
cods_unicos = df['historico_cod_ocor'].unique()
G_bip.add_nodes_from([f"COD_{c}" for c in cods_unicos], bipartite=1, tipo='ocorrencia')

# Arestas ponderadas: peso = frequência do par (rota, cod)
df_bip = (df.groupby(['ref_rota','historico_cod_ocor'])
            .size()
            .reset_index(name='peso'))

for _, row in df_bip.iterrows():
    G_bip.add_edge(row['ref_rota'], f"COD_{row['historico_cod_ocor']}",
                   peso=row['peso'])

print("━━━ GRAFO BIPARTIDO (ref_rota × cod_ocor) ━━━")
print(f"  Nós  : {G_bip.number_of_nodes():>6,}  (rotas={len(rotas_unicas)}, ocorrências={len(cods_unicos)})")
print(f"  Arestas: {G_bip.number_of_edges():>6,}")
print(f"  Densidade: {nx.density(G_bip):.6f}")


In [ ]:
# ═══════════════════════════════════════════════════════
# GRAFO 2 — LOGÍSTICO: DiGraph ponderado cidade→cidade
# ═══════════════════════════════════════════════════════
G_log = nx.DiGraph()

# Agrupamento: soma de ocorrências por (origem, destino, cod_ocor)
df_agg = (df.groupby(['origem','destino','historico_cod_ocor'])
            .agg(peso=('historico_cod_ocor','count'),
                 peso_medio_kg=('peso_real_kg','mean'),
                 valor_medio=('valor_mercadoria','mean'))
            .reset_index())

# Adiciona nós com atributo de hub (unidade_emissora)
hubs = set(df['unidade_emissora'].dropna().str.strip().str.upper().unique())
todas_cidades = set(df['origem'].unique()) | set(df['destino'].unique())

for cidade in todas_cidades:
    eh_hub = cidade in hubs
    G_log.add_node(cidade, eh_hub=eh_hub)

# Adiciona arestas; se já existe, soma o peso
for _, row in df_agg.iterrows():
    u, v = row['origem'], row['destino']
    if G_log.has_edge(u, v):
        G_log[u][v]['peso'] += row['peso']
    else:
        G_log.add_edge(u, v,
                       peso=row['peso'],
                       cod_ocor_principal=row['historico_cod_ocor'],
                       peso_medio_kg=row['peso_medio_kg'],
                       valor_medio=row['valor_medio'])

# Versão não-direcionada para métricas que exigem grafo simples
G_und = G_log.to_undirected()

print("━━━ GRAFO LOGÍSTICO (cidade → cidade) ━━━")
print(f"  Nós    : {G_log.number_of_nodes():>6,}")
print(f"  Arestas: {G_log.number_of_edges():>6,}")
print(f"  Densidade: {nx.density(G_log):.6f}")
is_conn = nx.is_weakly_connected(G_log)
print(f"  Fracamente conexo: {is_conn}")


## 4. Estatísticas Descritivas da Rede

In [ ]:
from networkx.algorithms.components import connected_components

def stats_grafo(G, nome, direcionado=False):
    """Retorna tabela de métricas estruturais do grafo."""
    n = G.number_of_nodes()
    m = G.number_of_edges()
    densid = nx.density(G)

    # Componente gigante (para não-direcionado ou conversão)
    Gu = G.to_undirected() if direcionado else G
    componentes = list(connected_components(Gu))
    maior_comp  = max(componentes, key=len)
    G_gc = Gu.subgraph(maior_comp).copy()

    diametro     = nx.diameter(G_gc)
    caminho_med  = nx.average_shortest_path_length(G_gc)
    clustering   = nx.average_clustering(Gu)
    graus        = [d for _, d in Gu.degree()]
    grau_medio   = np.mean(graus)
    grau_max     = max(graus)

    print(f"╔══════════════════════════════════════════════╗")
    print(f"║  Métricas — {nome:<32}║")
    print(f"╠══════════════════════════════════════════════╣")
    print(f"║  Nós                     : {n:>8,}          ║")
    print(f"║  Arestas                 : {m:>8,}          ║")
    print(f"║  Densidade               : {densid:>12.6f}      ║")
    print(f"║  Nº de componentes       : {len(componentes):>8,}          ║")
    print(f"║  Tamanho maior componente: {len(maior_comp):>8,}          ║")
    print(f"║  Diâmetro (comp. gigante): {diametro:>8,}          ║")
    print(f"║  Caminho médio           : {caminho_med:>12.4f}      ║")
    print(f"║  Coef. clustering médio  : {clustering:>12.4f}      ║")
    print(f"║  Grau médio              : {grau_medio:>12.4f}      ║")
    print(f"║  Grau máximo             : {grau_max:>8,}          ║")
    print(f"╚══════════════════════════════════════════════╝")
    return {
        'nos': n, 'arestas': m, 'densidade': densid,
        'componentes': len(componentes), 'diametro': diametro,
        'caminho_medio': caminho_med, 'clustering': clustering,
        'grau_medio': grau_medio, 'grau_max': grau_max
    }

stats_bip = stats_grafo(G_bip, "Grafo Bipartido")
print()
stats_log = stats_grafo(G_log, "Grafo Logístico", direcionado=True)


## 5. Métricas de Centralidade

In [ ]:
print("Calculando centralidades no Grafo Logístico (pode levar alguns segundos)…")

# Usa grafo não-direcionado para centralidades padrão
G_c = G_und  

degree_c     = nx.degree_centrality(G_c)
betweenness_c = nx.betweenness_centrality(G_c, weight='peso', normalized=True)
closeness_c  = nx.closeness_centrality(G_c)

# Eigenvector: requer componente gigante conexo
comp_gigante = max(nx.connected_components(G_c), key=len)
G_gc_und = G_c.subgraph(comp_gigante).copy()
eigen_c_gc = nx.eigenvector_centrality(G_gc_und, max_iter=1000, weight='peso')
# preenche 0 para nós fora da componente gigante
eigen_c = {n: eigen_c_gc.get(n, 0.0) for n in G_c.nodes()}

# ── Tabela consolidada ────────────────────────────────────────────────────────
df_centrality = pd.DataFrame({
    'cidade'         : list(degree_c.keys()),
    'degree'         : list(degree_c.values()),
    'betweenness'    : [betweenness_c[n] for n in degree_c],
    'closeness'      : [closeness_c[n]   for n in degree_c],
    'eigenvector'    : [eigen_c[n]        for n in degree_c],
}).sort_values('betweenness', ascending=False).reset_index(drop=True)

# Rank composto (média dos ranks)
for col in ['degree','betweenness','closeness','eigenvector']:
    df_centrality[f'rank_{col}'] = df_centrality[col].rank(ascending=False)
df_centrality['rank_composto'] = (
    df_centrality[['rank_degree','rank_betweenness','rank_closeness','rank_eigenvector']].mean(axis=1)
)
df_centrality = df_centrality.sort_values('rank_composto').reset_index(drop=True)

print("\n🏆 Top 15 cidades mais centrais (ranking composto):")
cols_show = ['cidade','degree','betweenness','closeness','eigenvector']
print(df_centrality[cols_show].head(15).to_string(index=False, float_format=lambda x: f"{x:.4f}"))


In [ ]:
# ── Visualização: Heatmap das centralidades (Top 20) ─────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 8))

metrics = ['degree', 'betweenness', 'closeness', 'eigenvector']
titles  = ['Degree\nCentrality', 'Betweenness\nCentrality',
           'Closeness\nCentrality', 'Eigenvector\nCentrality']
colors  = ['Blues_r', 'Reds_r', 'Greens_r', 'Purples_r']

top_n = 20
for ax, metric, title, cmap in zip(axes, metrics, titles, colors):
    top = df_centrality.nlargest(top_n, metric)[['cidade', metric]].reset_index(drop=True)
    sns.barh(top['cidade'], top[metric], ax=ax, palette=cmap, legend=False)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.invert_yaxis()
    ax.set_facecolor('#f8f9fa')

fig.suptitle('Métricas de Centralidade — Top 20 Cidades', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('centralidades_top20.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Figura salva: centralidades_top20.png")


## 6. Detecção de Comunidades — Algoritmo de Louvain

In [ ]:
# Louvain opera em grafo não-direcionado ponderado
partition = community_louvain.best_partition(G_und, weight='peso', random_state=42)

# Converte para DataFrame
df_comunidades = pd.DataFrame([
    {'cidade': node, 'comunidade': comm}
    for node, comm in partition.items()
])

n_comunidades = df_comunidades['comunidade'].nunique()
modularity    = community_louvain.modularity(partition, G_und, weight='peso')

print(f"Número de comunidades detectadas : {n_comunidades}")
print(f"Modularidade (Q)                 : {modularity:.4f}")
print()

# Tamanho de cada comunidade
tamanhos = df_comunidades.groupby('comunidade').size().sort_values(ascending=False)
print("Distribuição de tamanho das comunidades:")
print(tamanhos.rename_axis('comunidade').reset_index(name='cidades').head(15).to_string(index=False))

# Associa comunidade ao DataFrame de centralidade
df_centrality = df_centrality.merge(df_comunidades, on='cidade', how='left')


In [ ]:
# ── Visualização: Tamanho das comunidades ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Barras de tamanho
tamanhos_sorted = tamanhos.sort_values(ascending=False)
cores_comm = [cm.tab20(i / n_comunidades) for i in range(n_comunidades)]
ax1.bar(range(len(tamanhos_sorted)), tamanhos_sorted.values, color=cores_comm)
ax1.set_xlabel('Comunidade (ordenada por tamanho)')
ax1.set_ylabel('Número de cidades')
ax1.set_title('Tamanho das Comunidades Detectadas', fontweight='bold')

# Histograma
ax2.hist(tamanhos.values, bins=15, color='steelblue', edgecolor='white')
ax2.set_xlabel('Tamanho da comunidade')
ax2.set_ylabel('Frequência')
ax2.set_title('Distribuição de Tamanho das Comunidades', fontweight='bold')

plt.tight_layout()
plt.savefig('comunidades_tamanho.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Figura salva: comunidades_tamanho.png")


## 7. Distribuição de Grau e Comportamento Scale-Free

A hipótese de rede *scale-free* pressupõe que a distribuição de grau $P(k)$ segue uma lei de potência:  
$$P(k) \sim k^{-\gamma}$$  
Em escala log-log, isso se manifesta como uma reta com inclinação $-\gamma$.


In [ ]:
from scipy.stats import linregress

graus = dict(G_und.degree())
grau_vals = np.array(list(graus.values()))

# Distribuição empírica (normalizada como frequência relativa)
freq = Counter(grau_vals)
k_vals    = np.array(sorted(freq.keys()))
p_vals    = np.array([freq[k] / len(grau_vals) for k in k_vals])

# Ajuste linear em escala log-log (apenas k > 0)
mask = (k_vals > 0) & (p_vals > 0)
log_k = np.log10(k_vals[mask])
log_p = np.log10(p_vals[mask])
slope, intercept, r_value, p_value, std_err = linregress(log_k, log_p)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ─ Histograma linear ─────────────────────────────────────────────────────────
axes[0].hist(grau_vals, bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Grau')
axes[0].set_ylabel('Contagem')
axes[0].set_title('Distribuição de Grau (Escala Linear)', fontweight='bold')
axes[0].axvline(np.mean(grau_vals), color='red', linestyle='--', label=f'Média={np.mean(grau_vals):.1f}')
axes[0].legend()

# ─ Log-log com ajuste ────────────────────────────────────────────────────────
axes[1].scatter(log_k, log_p, color='steelblue', alpha=0.7, s=40, label='Dados')
fit_line = slope * log_k + intercept
axes[1].plot(log_k, fit_line, 'r--', linewidth=2,
             label=f'Ajuste: γ={-slope:.2f}\n$R^2$={r_value**2:.3f}')
axes[1].set_xlabel('log₁₀(Grau)')
axes[1].set_ylabel('log₁₀(P(k))')
axes[1].set_title('Distribuição Log-Log (Scale-Free?)', fontweight='bold')
axes[1].legend()

# ─ CCDF (Complementary CDF) — mais robusto para scale-free ───────────────────
graus_sorted = np.sort(grau_vals)[::-1]
ccdf = np.arange(1, len(graus_sorted)+1) / len(graus_sorted)
axes[2].loglog(graus_sorted, ccdf, '.', color='steelblue', alpha=0.6)
axes[2].set_xlabel('Grau k')
axes[2].set_ylabel('P(K ≥ k)')
axes[2].set_title('CCDF da Distribuição de Grau', fontweight='bold')

plt.suptitle('Análise da Distribuição de Grau — Grafo Logístico', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('distribuicao_grau.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Expoente γ estimado       : {-slope:.4f}")
print(f"R² do ajuste log-log      : {r_value**2:.4f}")
print(f"Grau mínimo / médio / max : {grau_vals.min()} / {grau_vals.mean():.2f} / {grau_vals.max()}")


## 8. Visualização da Rede

In [ ]:
# ── Parâmetros visuais ────────────────────────────────────────────────────────
# Tamanho do nó proporcional ao grau
degree_dict = dict(G_und.degree())
node_sizes  = np.array([degree_dict.get(n, 1) for n in G_und.nodes()])
node_sizes  = 50 + 600 * (node_sizes - node_sizes.min()) / (node_sizes.max() - node_sizes.min() + 1)

# Cor por comunidade
comm_map   = partition
n_comm     = len(set(comm_map.values()))
cmap_nodes = cm.get_cmap('tab20', n_comm)
node_colors = [cmap_nodes(comm_map.get(n, 0) / max(n_comm - 1, 1)) for n in G_und.nodes()]

# Layout: usa Kamada-Kawai (melhor qualidade para grafos médios)
print("Calculando layout (Kamada-Kawai)… aguarde.")
pos = nx.kamada_kawai_layout(G_und, weight='peso')

# Peso das arestas para espessura
pesos = [G_und[u][v].get('peso', 1) for u, v in G_und.edges()]
peso_norm = np.array(pesos, dtype=float)
peso_norm = 0.2 + 2.5 * (peso_norm - peso_norm.min()) / (peso_norm.max() - peso_norm.min() + 1)

fig, ax = plt.subplots(figsize=(20, 16))
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')

nx.draw_networkx_edges(G_und, pos, ax=ax,
                       width=peso_norm, alpha=0.3,
                       edge_color='#aaaaaa')

nx.draw_networkx_nodes(G_und, pos, ax=ax,
                       node_size=node_sizes,
                       node_color=node_colors,
                       alpha=0.9)

# Labels apenas para os nós mais centrais
top_nodes = df_centrality.nlargest(25, 'betweenness')['cidade'].tolist()
labels    = {n: n for n in G_und.nodes() if n in top_nodes}
nx.draw_networkx_labels(G_und, pos, labels=labels, ax=ax,
                        font_size=7, font_color='white', font_weight='bold')

ax.set_title('Rede Logística — Cidades por Comunidade\n'
             '(Tamanho ∝ grau | Cor = comunidade | Espessura ∝ frequência de ocorrências)',
             color='white', fontsize=14, fontweight='bold', pad=15)
ax.axis('off')

plt.tight_layout()
plt.savefig('grafo_logistico.png', bbox_inches='tight', dpi=150,
            facecolor='#0d1117')
plt.show()
print("✅ Figura salva: grafo_logistico.png")


## 9. Visualização do Grafo Bipartido (ref_rota × cod_ocor)

In [ ]:
# Seleciona as top-N rotas mais frequentes para visualização legível
top_rotas_freq = df['ref_rota'].value_counts().head(40).index.tolist()
nos_ocor = [f"COD_{c}" for c in df['historico_cod_ocor'].unique()]
nos_vis  = top_rotas_freq + nos_ocor

G_bip_sub = G_bip.subgraph(nos_vis).copy()

top_nodes_bip = {n for n, d in G_bip_sub.nodes(data=True) if d.get('bipartite') == 0}
pos_bip = nx.bipartite_layout(G_bip_sub, top_nodes_bip, scale=3, aspect_ratio=2.5)

pesos_bip  = [G_bip_sub[u][v].get('peso', 1) for u, v in G_bip_sub.edges()]
peso_b_max = max(pesos_bip)
widths_bip = [0.5 + 4.0 * (p / peso_b_max) for p in pesos_bip]

fig, ax = plt.subplots(figsize=(22, 10))
ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('#f0f4f8')

nx.draw_networkx_edges(G_bip_sub, pos_bip, ax=ax,
                       width=widths_bip, alpha=0.5, edge_color='#5577aa')

nx.draw_networkx_nodes(G_bip_sub, pos_bip, ax=ax,
                       nodelist=list(top_nodes_bip),
                       node_color='#4e79a7', node_size=300, alpha=0.9)
nx.draw_networkx_nodes(G_bip_sub, pos_bip, ax=ax,
                       nodelist=nos_ocor,
                       node_color='#e15759', node_size=700, alpha=0.95)

nx.draw_networkx_labels(G_bip_sub, pos_bip, ax=ax,
                        labels={n: n for n in nos_ocor},
                        font_size=9, font_color='white', font_weight='bold')
nx.draw_networkx_labels(G_bip_sub, pos_bip, ax=ax,
                        labels={n: n.replace(' ',  '\n') for n in top_nodes_bip},
                        font_size=5.5, font_color='#1a1a2e')

from matplotlib.patches import Patch
legend = [Patch(facecolor='#4e79a7', label='Rota (ref_rota)'),
          Patch(facecolor='#e15759', label='Código de Ocorrência')]
ax.legend(handles=legend, loc='lower right', fontsize=10)
ax.set_title(f'Grafo Bipartido — Top 40 Rotas × Códigos de Ocorrência\n'
             f'(Espessura da aresta ∝ frequência de ocorrências)',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('grafo_bipartido.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Figura salva: grafo_bipartido.png")


## 10. Exportação das Features de Rede

Todas as métricas calculadas são enriquecidas de volta ao dataset original,  
gerando um arquivo CSV pronto para ingestão nos modelos preditivos.


In [ ]:
# ── Monta DataFrame de features por cidade ────────────────────────────────────
df_feat_cidade = pd.DataFrame({
    'cidade'          : list(degree_c.keys()),
    'grau'            : [G_und.degree(n) for n in degree_c],
    'degree_c'        : list(degree_c.values()),
    'betweenness_c'   : [betweenness_c[n]  for n in degree_c],
    'closeness_c'     : [closeness_c[n]    for n in degree_c],
    'eigenvector_c'   : [eigen_c[n]         for n in degree_c],
    'comunidade'      : [partition.get(n, -1) for n in degree_c],
    'eh_hub'          : [G_log.nodes[n].get('eh_hub', False) if n in G_log.nodes else False
                         for n in degree_c],
})

# ── Join com o dataset original (por origem) ──────────────────────────────────
df_enriquecido = df.merge(
    df_feat_cidade.add_prefix('origem_'),
    left_on='origem', right_on='origem_cidade', how='left'
).merge(
    df_feat_cidade.add_prefix('destino_'),
    left_on='destino', right_on='destino_cidade', how='left'
)

# Remove colunas auxiliares duplicadas
cols_drop = [c for c in df_enriquecido.columns if c.endswith('_cidade')]
df_enriquecido.drop(columns=cols_drop, inplace=True, errors='ignore')

# Exporta
df_enriquecido.to_csv('banco_ia_features_rede.csv', index=False)

print(f"Dataset enriquecido salvo: banco_ia_features_rede.csv")
print(f"Shape: {df_enriquecido.shape}")
print()
print("Features de rede adicionadas ao dataset:")
novas_cols = [c for c in df_enriquecido.columns if c.startswith(('origem_', 'destino_'))]
for c in novas_cols:
    print(f"  • {c}")


## 11. Resumo Analítico Consolidado

In [ ]:
print("=" * 60)
print("   RESUMO ANALÍTICO — ANÁLISE DE REDES COMPLEXAS")
print("=" * 60)

print("\n📊 ESTRUTURA DA REDE LOGÍSTICA")
print(f"  Cidades (nós)        : {G_log.number_of_nodes()}")
print(f"  Conexões (arestas)   : {G_log.number_of_edges()}")
print(f"  Densidade            : {nx.density(G_log):.6f}")

gc = max(nx.weakly_connected_components(G_log), key=len)
print(f"  Componente gigante   : {len(gc)} cidades ({len(gc)/G_log.number_of_nodes()*100:.1f}%)")

print(f"\n🏘️ COMUNIDADES (Louvain)")
print(f"  Nº de comunidades    : {n_comunidades}")
print(f"  Modularidade Q       : {modularity:.4f}")
print(f"  Maior comunidade     : {tamanhos.max()} cidades")
print(f"  Menor comunidade     : {tamanhos.min()} cidades")

print(f"\n🎯 CIDADES MAIS CENTRAIS (Top 5 por Betweenness)")
top5 = df_centrality.nlargest(5, 'betweenness')[['cidade','betweenness','degree','comunidade']]
print(top5.to_string(index=False))

print(f"\n📈 DISTRIBUIÇÃO DE GRAU (Scale-Free)")
print(f"  Expoente γ estimado  : {-slope:.4f}")
print(f"  R² do ajuste         : {r_value**2:.4f}")
if r_value**2 > 0.7 and 2 <= -slope <= 3:
    print("  ➜ Comportamento SCALE-FREE confirmado (γ ∈ [2,3], R²>0.7)")
elif r_value**2 > 0.5:
    print("  ➜ Comportamento parcialmente scale-free")
else:
    print("  ➜ Distribuição não segue lei de potência clara")

print(f"\n💾 DATASET ENRIQUECIDO")
print(f"  Arquivo: banco_ia_features_rede.csv")
print(f"  Shape  : {df_enriquecido.shape}")
print(f"  Novas features de rede: {len(novas_cols)}")
print("=" * 60)
